# Assignment 9


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import wiener
from skimage.feature import local_binary_pattern

gray_img = cv2.imread('Sunflower_from_Silesia2.jpg', cv2.IMREAD_GRAYSCALE)


## Question 1: Local Binary Pattern (LBP)


In [ ]:
def compute_lbp(image):
    h, w = image.shape
    lbp_img = np.zeros((h - 2, w - 2), dtype=np.uint8)

    offsets = [
        (-1, -1, 0),
        (-1,  0, 1),
        (-1,  1, 2),
        ( 0,  1, 3),
        ( 1,  1, 4),
        ( 1,  0, 5),
        ( 1, -1, 6),
        ( 0, -1, 7)
    ]

    center = image[1:-1, 1:-1]
    for dy, dx, power in offsets:
        neighbor = image[1 + dy : h - 1 + dy, 1 + dx : w - 1 + dx]
        bit = (neighbor >= center).astype(np.uint8)
        lbp_img += bit * (2 ** power)

    return lbp_img

lbp_result = compute_lbp(gray_img)
lbp_hist, _ = np.histogram(lbp_result.ravel(), bins=256, range=[0, 256])

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title('Original Image')
plt.imshow(gray_img, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('Local Binary Pattern (LBP) Image')
plt.imshow(lbp_result, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('LBP Feature Histogram')
plt.bar(range(256), lbp_hist, color='purple', width=1)
plt.xlim([0, 255])
plt.xlabel('LBP Code')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()


## Question 2: Denoising Filters (3x3 and 5x5)


In [ ]:
np.random.seed(42)

gauss_noise = np.random.normal(0, 25, gray_img.shape)
noisy_gauss = np.uint8(np.clip(gray_img.astype(np.float64) + gauss_noise, 0, 255))

speckle_noise = np.random.normal(0, 0.15, gray_img.shape)
noisy_speckle = np.uint8(np.clip(gray_img.astype(np.float64) * (1.0 + speckle_noise), 0, 255))

def compute_psnr(clean, denoised):
    mse = np.mean((clean.astype(np.float64) - denoised.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 10.0 * np.log10((255.0 ** 2) / mse)


In [ ]:
def run_denoising(noisy_img, clean_img, noise_name):
    results = {}

    for k in [3, 5]:
        w_out = wiener(noisy_img.astype(np.float64), (k, k))
        w_img = np.uint8(np.clip(w_out, 0, 255))

        med_img = cv2.medianBlur(noisy_img, k)
        gauss_filt = cv2.GaussianBlur(noisy_img, (k, k), sigmaX=1.0)
        bilat_filt = cv2.bilateralFilter(noisy_img, d=k, sigmaColor=50, sigmaSpace=50)

        results[k] = {
            'Wiener': (w_img, compute_psnr(clean_img, w_img)),
            'Median': (med_img, compute_psnr(clean_img, med_img)),
            'Gaussian': (gauss_filt, compute_psnr(clean_img, gauss_filt)),
            'Bilateral': (bilat_filt, compute_psnr(clean_img, bilat_filt))
        }

    plt.figure(figsize=(16, 10))
    plt.suptitle(f'Denoising Results for {noise_name} Noise', fontsize=15)

    filters = ['Wiener', 'Median', 'Gaussian', 'Bilateral']
    for idx, f_name in enumerate(filters):
        img_f, psnr = results[3][f_name]
        plt.subplot(2, 4, idx + 1)
        plt.title(f'{f_name} (3x3)\nPSNR: {psnr:.2f} dB')
        plt.imshow(img_f, cmap='gray')
        plt.axis('off')

    for idx, f_name in enumerate(filters):
        img_f, psnr = results[5][f_name]
        plt.subplot(2, 4, idx + 5)
        plt.title(f'{f_name} (5x5)\nPSNR: {psnr:.2f} dB')
        plt.imshow(img_f, cmap='gray')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

run_denoising(noisy_gauss, gray_img, 'Gaussian')
run_denoising(noisy_speckle, gray_img, 'Speckle')
